TOOLS

models can call tools to perform action such as fetching or searching a daata from adatabse , searching the web or running a code 
tools are piring of
  1. A schema, including the name of the tool,  a description and / or argument definitons (often a json schema)
  2. a function or a coroutine to execute

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("why do parrots talk")
response

AIMessage(content='Parrots are among the most vocal and socially sophisticated birds in the world, and their “talking” is a fascinating blend of biology, behavior, and learning. Here’s why they do it:\n\n### 1. **Vocal Learning – The Key to Mimicry**\n\n- **Syrinx Structure**: Parrots have a highly flexible syrinx (the bird equivalent of a human larynx) that lets them produce a wide range of sounds. The syrinx is located at the base of the trachea and can be controlled with remarkable precision.\n- **Brain Circuitry**: They possess a specialized set of brain nuclei (the “song system”) that supports vocal learning. Unlike many birds that simply repeat innate songs, parrots actively learn and modify sounds from their environment.\n- **Learning Process**: As chicks, parrots listen to the calls of their parents and peers. They then practice producing those sounds, gradually refining them. This process is similar to how humans learn language.\n\n### 2. **Social Function – Communication and 

In [4]:
from langchain.tools import tool

@tool
def get_weather(city:str)->str:
  """get the weather for a partcular city""" #this is very important as each tool nneeds its own description
  return f"the weather in the {city} is sunny"

model_with_tool=model.bind_tools([get_weather])

In [5]:
response=model_with_tool.invoke("what is the weather in the new york?")
for toolcall in response.tool_calls:
  #view tools calls done by the model
  print(f"tool : {toolcall['name']}")
  print(f"args : {toolcall['args']}")

tool : get_weather
args : {'city': 'New York'}


TOOL EXECUTOIN LOOP

In [6]:
# Step 1: Model generates tool calls

messages = [{"role": "user", "content": "What's the weather in Boston?"}]

ai_msg = model_with_tool.invoke(messages)

messages.append(ai_msg)


# Step 2: Execute tools and collect results

for tool_call in ai_msg.tool_calls:

    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)

    messages.append(tool_result)


# Step 3: Pass results back to model for final response

final_response = model_with_tool.invoke(messages)

print(final_response.text)

It’s sunny in Boston today! 🌞 If you’re heading out, a light jacket should do—temperatures are usually in the 60s‑70s (°F) during the day. Let me know if you need anything else!
